# Architecture Matching and Hyperparameter Search

This notebook compares KAN and MLP architectures and prepares candidate configurations for hyperparameter optimization.

In [26]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Search Spaces and Candidate Configurations

In [27]:
from itertools import product

# Candidate values for the KAN search.
# Five optimization hyperparameters:
# hidden_layers, hidden_units, grid_size, spline_order, learning_rate
kan_search_space = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 25, 35],
    "grid_size": [3, 5, 7],
    "spline_order": [2, 3, 4],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

# Candidate values for the MLP search.
# Four optimization hyperparameters:
# hidden_layers, hidden_units, activation, learning_rate
mlp_search_space = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [25, 50, 75],
    "activation": ["Tanh", "SiLU", "Sigmoid"],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

kan_configurations = [
    dict(zip(kan_search_space, values))
    for values in product(*kan_search_space.values())
]

mlp_configurations = [
    dict(zip(mlp_search_space, values))
    for values in product(*mlp_search_space.values())
]

print(f"KAN candidate configurations: {len(kan_configurations)}")
print(f"MLP candidate configurations: {len(mlp_configurations)}")

print("\nExample KAN configuration:")
print(kan_configurations[0])

print("\nExample MLP configuration:")
print(mlp_configurations[0])

KAN candidate configurations: 243
MLP candidate configurations: 81

Example KAN configuration:
{'hidden_layers': 1, 'hidden_units': 15, 'grid_size': 3, 'spline_order': 2, 'learning_rate': 0.0001}

Example MLP configuration:
{'hidden_layers': 1, 'hidden_units': 25, 'activation': 'Tanh', 'learning_rate': 0.0001}


In [31]:
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

kan_config_path = data_dir / "kan_configurations.json"
mlp_config_path = data_dir / "mlp_configurations.json"

with kan_config_path.open("w", encoding="utf-8") as file:
    json.dump(kan_configurations, file, indent=2)

with mlp_config_path.open("w", encoding="utf-8") as file:
    json.dump(mlp_configurations, file, indent=2)

print(f"Saved {len(kan_configurations)} KAN configurations to {kan_config_path}")
print(f"Saved {len(mlp_configurations)} MLP configurations to {mlp_config_path}")

Saved 243 KAN configurations to data/kan_configurations.json
Saved 81 MLP configurations to data/mlp_configurations.json


## Architecture Matching

In [28]:
# -------------------------
# Setup & Configurations
# -------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

input_shape = (1, 2)

kan_depths = [1, 2, 3]
kan_widths = [15, 25, 35]
kan_grids = [5]
spline_order = 3

# MLP widths to test
mlp_sweep_widths = range(10, 150, 1)


# -------------------------
# Storage
# -------------------------
kan_labels = []

kan_params_list = []
mlp_matched_params_list = []

mlp_matched_widths = []


 


for depth in kan_depths:

    for units in kan_widths:

        for grid in kan_grids:

            # -------------------------------------------------
            # Build KAN
            # -------------------------------------------------
            try:

                model_kan, _ = build_models_KAN(
                    device,
                    hidden_layers=depth,
                    hidden_units=units,
                    grid_size=grid,
                    spline_order=spline_order,
                )

            except TypeError:

                model_kan, _ = build_models_KAN(
                    device,
                    hidden_units=units,
                    grid_size=grid,
                    spline_order=spline_order,
                )


            # -------------------------------------------------
            # Calculate KAN parameters
            # -------------------------------------------------
            kan_flops, _, kan_params = calculate_flops(
                model_kan,
                input_shape=input_shape,
                print_results=False,
                print_detailed=False,
                output_as_string=False,
            )


            # -------------------------------------------------
            # Find MLP with closest number of parameters
            # -------------------------------------------------
            best_width = None
            best_mlp_params = None
            min_difference = float("inf")


            for m_width in mlp_sweep_widths:

                model_mlp, _ = build_models(
                    device,
                    hidden_layers=depth,
                    hidden_units=m_width,
                )

                mlp_flops, _, mlp_params = calculate_flops(
                    model_mlp,
                    input_shape=input_shape,
                    print_results=False,
                    print_detailed=False,
                    output_as_string=False,
                )


                difference = abs(
                    mlp_params - kan_params
                )


                if difference < min_difference:

                    min_difference = difference

                    best_width = m_width

                    best_mlp_params = mlp_params


            # -------------------------------------------------
            # Store results
            # -------------------------------------------------
            kan_labels.append(
                f"L={depth}\nN={units}"
            )

            kan_params_list.append(
                kan_params
            )

            mlp_matched_widths.append(
                best_width
            )

            mlp_matched_params_list.append(
                best_mlp_params
            )


 

## Parameter-Matching Results

In [29]:
# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------
print("\n")
print("=" * 75)
print("PARAMETER-MATCHED ARCHITECTURES")
print("=" * 75)

print(
    f"{'KAN':<12} | "
    f"{'KAN Params':>12} | "
    f"{'MLP Width':>10} | "
    f"{'MLP Params':>12} | "
    f"{'Difference':>12}"
)

print("-" * 75)


for label, kan_p, mlp_w, mlp_p in zip(
    kan_labels,
    kan_params_list,
    mlp_matched_widths,
    mlp_matched_params_list,
):

    difference = abs(
        kan_p - mlp_p
    )

    print(
        f"{label.replace(chr(10), ', '):<12} | "
        f"{kan_p:>12,} | "
        f"{mlp_w:>10} | "
        f"{mlp_p:>12,} | "
        f"{difference:>12,}"
    )



PARAMETER-MATCHED ARCHITECTURES
KAN          |   KAN Params |  MLP Width |   MLP Params |   Difference
---------------------------------------------------------------------------
L=1, N=15    |          450 |         19 |          457 |            7
L=1, N=25    |          750 |         25 |          751 |            1
L=1, N=35    |        1,050 |         30 |        1,051 |            1
L=2, N=15    |        2,700 |         35 |        2,661 |           39
L=2, N=25    |        7,000 |         58 |        7,077 |           77
L=2, N=35    |       13,300 |         80 |       13,281 |           19
L=3, N=15    |        4,950 |         39 |        4,837 |          113
L=3, N=25    |       13,250 |         65 |       13,131 |          119
L=3, N=35    |       25,550 |         91 |       25,481 |           69
